# Diabetes Risk Prediction Using Machine Learning
### Academic Machine Learning Mini-Project

> **Educational & Medical Disclaimer:**
> This notebook and project are developed solely for academic and educational purposes. The predictions generated by the models are based on statistical patterns within the training dataset and must **never** be used as medical advice, clinical diagnoses, or clinical decision support tools.

## 1. Project Introduction & Objectives
The goal of this project is to build, evaluate, and compare multiple supervised machine learning classifiers to predict diabetes risk (the binary `Outcome` variable) from diagnostic and physiological measures.

### Key Objectives:
1. Perform comprehensive exploratory data analysis (EDA).
2. Identify and handle biologically impossible zero values without causing data leakage.
3. Implement leak-free preprocessing pipelines (imputation + scaling) using `scikit-learn`.
4. Train and compare 4 candidate models: Logistic Regression, K-Nearest Neighbors, Decision Tree, and Random Forest.
5. Use 5-Fold Stratified Cross-Validation for robust performance assessment.
6. Conduct hyperparameter tuning using `GridSearchCV`.
7. Evaluate the final model on held-out test data and extract feature importance.

## 2. Import Libraries
Import standard scientific computing, visualization, and machine learning libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.autolayout': True})
print("All required libraries successfully loaded.")

## 3. Load Dataset
Load the public Pima Indians Diabetes dataset from the project's data directory.

In [ ]:
df = pd.read_csv("../data/diabetes.csv")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 4. Dataset Overview & Data Quality Assessment
Inspect data types, missing records, duplicates, and summary statistics.

In [ ]:
print("--- Data Types ---")
print(df.dtypes)
print("\n--- Duplicate Records ---")
print("Duplicate rows:", df.duplicated().sum())
print("\n--- Explicit Null Values ---")
print(df.isnull().sum())
print("\n--- Summary Statistics ---")
df.describe().T

## 5. Identifying Biologically Implausible Zero Values
In physiological metrics, a living patient cannot have 0 for Glucose, Blood Pressure, Skin Thickness, Insulin, or BMI.
These 0 values represent missing records that must be addressed.
However, for `Pregnancies`, 0 is completely legitimate (indicating nulliparous subjects).

In [ ]:
zero_counts = (df == 0).sum()
print("Zero counts per feature:")
print(zero_counts)

zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nFeatures with biologically implausible zeros requiring median imputation:", zero_cols)

## 6. Exploratory Data Analysis (EDA)
Visualize target class distribution, feature distributions, and correlations.

In [ ]:
# Target Class Distribution
plt.figure(figsize=(6, 4))
counts = df["Outcome"].value_counts()
ax = sns.barplot(x=counts.index, y=counts.values, palette=["#2b5c8f", "#d95f02"])
plt.title("Target Class Distribution (Outcome)", weight="bold")
plt.xlabel("Class Label (0: Negative, 1: Positive)")
plt.ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, weight='bold')
plt.show()

# Correlation Heatmap
plt.figure(figsize=(8.5, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", mask=mask, linewidths=0.5)
plt.title("Feature Correlation Heatmap", weight="bold")
plt.show()

## 7. Feature and Target Definition (X and y)
Separate predictive features from the target variable in a stable, documented order.

In [ ]:
feature_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
X = df[feature_cols]
y = df["Outcome"]
print("Feature matrix shape:", X.shape)
print("Target vector shape :", y.shape)

## 8. Train/Test Split (Preventing Data Leakage)
Partition the data into 80% training and 20% testing sets using stratification.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} rows | Held-out Test set: {X_test.shape[0]} rows")
print("Train class ratio:", np.round(y_train.value_counts(normalize=True).values, 4))
print("Test class ratio :", np.round(y_test.value_counts(normalize=True).values, 4))

## 9. Preprocessing Pipeline Construction
We use `ColumnTransformer` to handle imputation and standardization strictly on training data.

In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
normal_cols = ["Pregnancies", "DiabetesPedigreeFunction", "Age"]

preprocessor = ColumnTransformer(
    transformers=[
        ("zero_impute", Pipeline([
            ("imputer", SimpleImputer(missing_values=0, strategy="median")),
            ("scaler", StandardScaler())
        ]), zero_cols),
        ("normal", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), normal_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)
print("Leak-free Preprocessing ColumnTransformer ready.")

## 10. Baseline Model Training & 5-Fold Stratified Cross-Validation
Train and evaluate 4 algorithms across 5 folds on the training data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
}

cv_summary = {}
for name, clf in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["accuracy", "precision", "recall", "f1", "roc_auc"])
    cv_summary[name] = {
        "CV Accuracy Mean": np.mean(scores["test_accuracy"]),
        "CV Accuracy Std": np.std(scores["test_accuracy"]),
        "CV F1 Mean": np.mean(scores["test_f1"]),
        "CV ROC-AUC Mean": np.mean(scores["test_roc_auc"])
    }
pd.DataFrame(cv_summary).T

## 11. Hyperparameter Tuning (GridSearchCV)
Tune candidate models using 5-fold cross-validation on `X_train`.

In [ ]:
lr_pipe = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(random_state=42, max_iter=1000))])
lr_grid = GridSearchCV(
    lr_pipe,
    param_grid={"classifier__C": [0.01, 0.1, 1.0, 10.0], "classifier__class_weight": [None, "balanced"]},
    cv=cv, scoring="f1"
)
lr_grid.fit(X_train, y_train)
print("Tuned LR Best CV F1:", round(lr_grid.best_score_, 4))
print("Tuned LR Best Params:", lr_grid.best_params_)

## 12. Final Model Evaluation on Held-Out Test Set
Evaluate the final selected model once on the held-out test data.

In [ ]:
final_pipe = Pipeline([("preprocessor", preprocessor), ("classifier", DecisionTreeClassifier(random_state=42, max_depth=5))])
final_pipe.fit(X_train, y_train)
y_test_pred = final_pipe.predict(X_test)
y_test_proba = final_pipe.predict_proba(X_test)[:, 1]

print(f"Test Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Test Recall   : {recall_score(y_test, y_test_pred):.4f}")
print(f"Test F1-Score : {f1_score(y_test, y_test_pred):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, y_test_proba):.4f}")
print("\n--- Classification Report ---\n", classification_report(y_test, y_test_pred))
print("--- Confusion Matrix ---\n", confusion_matrix(y_test, y_test_pred))

## 13. Feature Importance / Model Interpretation
Inspect the relative influence of features in the trained classifier.
**Note:** Feature importance indicates statistical correlation within this model, not medical causation.

In [ ]:
clf = final_pipe.named_steps["classifier"]
transformed_features = zero_cols + normal_cols
imp_series = pd.Series(clf.feature_importances_, index=transformed_features).sort_values(ascending=True)

plt.figure(figsize=(7, 4.5))
imp_series.plot(kind="barh", color="#2b5c8f", edgecolor="#1a3857")
plt.title("Features Most Influential in Trained Classifier\n(Statistical Correlation, Not Causation)", weight="bold")
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.show()

## 14. Conclusion & Limitations
### Conclusions:
- Machine learning algorithms detected meaningful statistical patterns in predicting the positive outcome from physiological inputs.
- Decision Tree and Tuned Logistic Regression delivered strong balance between Precision and Recall.
- Strict leakage prevention ensured that test evaluation metrics reflect realistic generalization.

### Known Limitations:
1. **Sample Size:** 768 total records from the Pima dataset is modest.
2. **Population-Specific:** The cohort consists exclusively of adult females of Pima Indian heritage; results cannot be generalized to broader global demographics.
3. **Educational Scope:** This project is designed strictly for academic study and is not clinically validated for diagnostic use.